# 09 딥러닝 예측 모델 (Phase 1 — 3/3)

**12조건**(SBC 8 + ML 4, 2-type E·C) × **LSTM, Autoformer, N-HiTS, iTransformer**

07·08 결과와 합쳐 **10모델 중 조건별 Best**(MAPE) 선정

## 구현 (공식 모델 코드)

| 모델 | 출처 |
|------|------|
| Autoformer, iTransformer | [thuml/Time-Series-Library](https://github.com/thuml/Time-Series-Library) submodule → `tslib_adapter.py` |
| N-HiTS, LSTM | [Nixtla/neuralforecast](https://github.com/Nixtla/neuralforecast) → `neuralforecast_adapter.py` |

- 조건(type×cluster) 내 **family 다변량 패널** 1회 학습 (SKU별 반복 학습 아님)
- lookback=**26**, horizon=**13주**(test), `DL_EPOCHS=60`(GPU), iTransformer는 **Optuna 튜닝**

> **논문 vs 실습:** 축소 데이터(66 시계열)에서는 SBA·ARIMA·N-HiTS 등 단순 모델도 경쟁적이라, 논문(iTransformer 강세)과 달리 대형 트랜스포머는 이 규모에서 학습되지 않습니다. iTransformer는 median MAPE **51.6으로 8위**, Autoformer는 **73.1로 10위**입니다. 반면 파라미터가 적은 **LSTM(42.6, 2위)** 과 **N-HiTS(46.4, 4위)** 는 XGBoost·RF보다 앞섭니다 — 밀린 것은 딥러닝 전체가 아니라 어텐션 기반 트랜스포머 계열입니다.

### ⓪ 환경 설정

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.device import device_label
from utils.experiment_data import load_forecast_frames
from utils.phase_experiments import (
    STAT_MODELS, ML_MODELS, DL_MODELS, RANK_METRIC,
    run_phase1_all, summarize_phase1, merge_phase1_best,
    build_global_embedding_cache, run_phase2_all, summarize_phase2,
    pick_final_per_condition,
)

print('Torch device:', device_label())
df, feat_df = load_forecast_frames()
print('시계열:', df.groupby(['type', 'family']).ngroups)
print('학습 <=', TRAIN_WEEK_MAX, '| 검증:', VAL_WEEKS)
print('Best 선정 기준:', RANK_METRIC.upper())


Torch device: cuda (NVIDIA GeForce RTX 5090, 32GB)
시계열: 66
학습 <= 201707 | 검증: [201708, 201709, 201710, 201711, 201712, 201713, 201714, 201715, 201716, 201717, 201718, 201719, 201720]
Best 선정 기준: MAPE


### ① DL 모델 실험

In [2]:
# Phase 1 — DL 4종: SBC 20조건 + ML 20조건
cache = DATA_PROCESSED / 'phase1_dl_results.parquet'
if cache.exists():
    results = pd.read_parquet(cache)
    summary, best = summarize_phase1(results)
    print('캐시 로드 |', len(results), 'rows')
else:
    sbc = run_phase1_all(df, feat_df, 'SBC_CLUSTER', 'SBC', models=DL_MODELS)
    ml = run_phase1_all(df, feat_df, 'ML_CLUSTER', 'ML', models=DL_MODELS)
    results = pd.concat([sbc, ml], ignore_index=True)
    summary, best = summarize_phase1(results)
    results.to_parquet(cache, index=False)
    summary.to_csv(DATA_PROCESSED / 'phase1_dl_results_summary.csv', index=False)
    best.to_csv(DATA_PROCESSED / 'phase1_dl_results_best.csv', index=False)
    print('완료 |', len(results), 'rows')

display(best.sort_values(['cluster_scheme', 'type', 'cluster']))
print('\n=== 알고리즘별 평균', RANK_METRIC.upper(), '===')
print(results.groupby(['cluster_scheme', 'model'])[RANK_METRIC].mean().unstack('cluster_scheme').round(2))


Phase1 SBC:   0%|          | 0/8 [00:00<?, ?it/s]

Seed set to 42


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


You are using a CUDA device ('NVIDIA GeForce RTX 5090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


...

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


C:\Users\kjh\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


완료 | 528 rows


,cluster_scheme,type,cluster,best_model,mae_mean,rmse_mean,best_mape,mase_mean
1,ML,C,1,LSTM,978.721605,1825.493789,40.281871,1.796780
6,ML,C,2,N-HiTS,31041.467338,48704.903294,28.591267,2.189210
9,ML,E,1,LSTM,658.715388,1139.671203,53.772767,2.063632
13,ML,E,2,LSTM,13055.796406,23557.874132,24.227925,1.745374
19,SBC,C,1,iTransformer,6053.119415,9931.637495,38.149790,2.049564
22,SBC,C,2,N-HiTS,803.278985,1776.432415,35.112289,1.668610
26,SBC,C,3,N-HiTS,720.136017,1276.584199,32.898739,0.454482
29,SBC,C,4,LSTM,18.677950,41.936900,44.880260,0.672614
34,SBC,E,1,N-HiTS,1956.553403,3710.667796,42.673543,1.550132
37,SBC,E,2,LSTM,563.217163,1088.439120,55.341612,2.472732



=== 알고리즘별 평균 MAPE ===
cluster_scheme      ML     SBC
model                         
Autoformer      141.56  128.24
LSTM             45.52   44.44
N-HiTS          107.38   89.88
iTransformer    191.19   92.27


### ② Phase 1 통합 Best (10모델)

In [3]:
stat = pd.read_parquet(DATA_PROCESSED / 'phase1_stat_results.parquet')
ml = pd.read_parquet(DATA_PROCESSED / 'phase1_ml_results.parquet')
dl = pd.read_parquet(DATA_PROCESSED / 'phase1_dl_results.parquet')

phase1_all = pd.concat([stat, ml, dl], ignore_index=True)
phase1_summary, phase1_best = merge_phase1_best(stat, ml, dl)
phase1_all.to_parquet(DATA_PROCESSED / 'phase1_all_results.parquet', index=False)
phase1_summary.to_csv(DATA_PROCESSED / 'phase1_summary.csv', index=False)
phase1_best.to_csv(DATA_PROCESSED / 'phase1_best_per_condition.csv', index=False)
print('Phase1 통합 Best |', len(phase1_best), '조건')
display(phase1_best.sort_values(['cluster_scheme', 'type', 'cluster']))


Phase1 통합 Best | 12 조건


,cluster_scheme,type,cluster,best_model,mae_mean,rmse_mean,best_mape,mase_mean
2,ML,C,1,LSTM,978.721605,1825.493789,40.281871,1.796780
14,ML,C,2,Prophet,29321.046119,46593.903215,26.679884,2.172205
20,ML,E,1,ARIMA,711.689891,1193.188452,42.765207,1.941735
38,ML,E,2,XGBoost,14612.161896,22322.276167,23.696725,1.945015
44,SBC,C,1,Prophet,5710.249524,9175.945334,35.181773,2.230524
50,SBC,C,2,ARIMA,1603.653583,1932.700734,28.708055,1.940871
63,SBC,C,3,N-HiTS,720.136017,1276.584199,32.898739,0.454482
72,SBC,C,4,LSTM,18.677950,41.936900,44.880260,0.672614
80,SBC,E,1,ARIMA,2290.985782,3917.372707,38.655918,1.685987
90,SBC,E,2,ARIMA,554.776594,934.173392,40.656822,2.466498
